#  MARL for Energy Control on CityLearn Training Notebook

**Experiment:** Full 2 × 3 training matrix on the CityLearn Challenge 2023 US-climate dataset.
**Run:** Run Seeds independently by changing the seed and run_index values

| Algorithms | Reward designs |
|-----------|---------------|
| IPPO | flat\_shared |
| MAPPO | local\_individual |
| | uae\_weighted |

Six conditions are trained in sequence. A condition is skipped automatically if its
checkpoint already exists, making the notebook safe to re-run after interruption.


## 0 · Dependencies

In [ ]:
# Run this cell first. It checks the environment and installs only what is missing.
import sys
import subprocess
import importlib

REQUIRED = {
    "citylearn":  "citylearn==2.1.2",
    "torch":      "torch>=2.0",
    "numpy":      "numpy>=1.24,<2.0",
    "pandas":     "pandas==1.3.5",
    "gymnasium":  "gymnasium==0.29.1",
    "matplotlib": "matplotlib>=3.7",
    "sklearn":    "scikit-learn==1.0.2",
    "ipython":    "ipython>=8.0",
}

print(f"Python {sys.version}")
if sys.version_info < (3, 9):
    raise RuntimeError("Python 3.9+ is required. Please switch kernels.")
if sys.version_info >= (3, 12):
    raise RuntimeError(
        "This notebook is intended for Python 3.10 or 3.11. "
        "Please switch to a compatible Jupyter kernel."
    )

missing = []
for mod_name, pip_spec in REQUIRED.items():
    import_name = "sklearn" if mod_name == "sklearn" else mod_name
    spec = importlib.util.find_spec(import_name)
    if spec is None:
        missing.append(pip_spec)

if missing:
    print("\nInstalling missing packages:")
    for pkg in missing:
        print("  ", pkg)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    print("\nInstallation complete. Re-checking imports...\n")

print("Installed package versions:")
for mod_name, _ in REQUIRED.items():
    import_name = "sklearn" if mod_name == "sklearn" else mod_name
    mod = importlib.import_module(import_name)
    ver = getattr(mod, "__version__", "?")
    print(f"  {mod_name:<12} {ver}")

try:
    import torch
    _ = torch.zeros(1)
    print(f"\nTorch device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
except Exception as e:
    raise ImportError(f"PyTorch import failed after install: {e}")

print("\nAll dependencies OK.")


## 1 · Configuration

`RUN_INDEX` is the run label.
`SEED` is the actual random seed used for all stochastic operations.
Output paths use `seed(run_index)` and not `seed(value)` so run numbering stays consistent.


In [ ]:
from pathlib import Path

RUN_INDEX  = 3          # third independent run
SEED       = 7          # actual random seed
RUN_LABEL  = f"seed{RUN_INDEX}"           # "seed3"
CKPT_SUFFIX = f"_250k_{RUN_LABEL}"        # "_250k_seed3"

DATASET_NAME  = "citylearn_challenge_2023_phase_3_1"
RESULTS_DIR   = Path(f"results_250k_{RUN_LABEL}")
MODELS_DIR    = Path("models")
ARTIFACTS_DIR = Path("artifacts")

CONDITIONS = [
    ("ippo",  "flat_shared"),
    ("ippo",  "local_individual"),
    ("ippo",  "uae_weighted"),
    ("mappo", "flat_shared"),
    ("mappo", "local_individual"),
    ("mappo", "uae_weighted"),
]

for d in [RESULTS_DIR / "conditions", RESULTS_DIR / "summary",
          MODELS_DIR, ARTIFACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def ckpt_path(algo: str, reward_mode: str) -> Path:
    return MODELS_DIR / f"{algo}_{reward_mode}{CKPT_SUFFIX}.pt"

def cond_dir(algo: str, reward_mode: str) -> Path:
    d = RESULTS_DIR / "conditions" / f"{algo}_{reward_mode}"
    d.mkdir(parents=True, exist_ok=True)
    return d

print(f"Run label : {RUN_LABEL}")
print(f"Seed      : {SEED}")
print(f"Results   : {RESULTS_DIR}/")
print(f"Conditions: {[f'{a}_{r}' for a,r in CONDITIONS]}")


## 2 · Imports and Helpers

In [ ]:
from __future__ import annotations

import math
import random
import warnings
from dataclasses import dataclass, field
from typing import Dict, Iterable, List, Mapping, Optional, Tuple, Union

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from citylearn.citylearn import CityLearnEnv
from citylearn.data import DataSet
from citylearn.reward_function import RewardFunction
from IPython.display import Image, display
from torch.distributions import Normal

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print("Device:", DEVICE)


## 3 · Reward Functions

Three reward designs are supported:

| Name | Description |
|------|-------------|
| `flat_shared` | Equal weights (0.10 each), shared district-level signal — matches the benchmark |
| `local_individual` | Same weights, per-building signal — each agent optimises its own KPIs |
| `uae_weighted` | Higher weight on discomfort and electricity, shared — reflects UAE cooling dominance |

All weight vectors sum to 0.40 so reward magnitudes are comparable across modes.
Internal ramping memory is reset between episodes via `reset_memory()`.


In [ ]:
FLAT_WEIGHTS: Dict[str, float] = {
    "discomfort":  0.10,
    "electricity": 0.10,
    "ramping":     0.10,
    "solar":       0.10,
}

UAE_WEIGHTS: Dict[str, float] = {
    "discomfort":  0.140,  # 0.35 / 2.5 — reflects 70% UAE cooling load vs 19% US
    "electricity": 0.120,  # 0.30 / 2.5
    "ramping":     0.060,  # 0.15 / 2.5
    "solar":       0.080,  # 0.20 / 2.5
}


class ProjectReward(RewardFunction):
    def __init__(
        self,
        env_metadata=None,
        weights: Optional[Dict[str, float]] = None,
        shared: bool = True,
        **_,
    ):
        super().__init__(env_metadata)
        self.weights = dict(FLAT_WEIGHTS if weights is None else weights)
        self.shared = shared
        self._prev_district_elec: Optional[float] = None
        self._prev_local_elec: Optional[List[float]] = None

    def calculate(self, observations) -> List[float]:
        comps = [self._components(o) for o in observations]

        d_elec = float(sum(c["electricity"] for c in comps))
        d_ramp = (0.0 if self._prev_district_elec is None
                  else abs(d_elec - self._prev_district_elec))
        self._prev_district_elec = d_elec

        l_elec = [c["electricity"] for c in comps]
        if self._prev_local_elec is None:
            l_ramp = [0.0] * len(l_elec)
        else:
            l_ramp = [abs(a - b) for a, b in zip(l_elec, self._prev_local_elec)]
        self._prev_local_elec = l_elec

        if self.shared:
            dc = {
                "discomfort":  float(sum(c["discomfort"] for c in comps)),
                "electricity": d_elec,
                "ramping":     d_ramp,
                "solar":       float(sum(c["solar"] for c in comps)),
            }
            r = -self._wsum(dc)
            return [r] * len(observations)

        return [
            -self._wsum({**c, "ramping": l_ramp[i]})
            for i, c in enumerate(comps)
        ]

    def reset_memory(self) -> None:
        self._prev_district_elec = None
        self._prev_local_elec = None

    def _components(self, obs: Mapping) -> Dict[str, float]:
        net = float(obs.get("net_electricity_consumption", 0.0))
        imported = max(net, 0.0)
        solar = max(float(obs.get("solar_generation", 0.0)), 0.0)
        return {
            "discomfort":  self._discomfort(obs),
            "electricity": imported,
            "ramping":     0.0,
            "solar":       imported if solar > 0.0 else 0.0,
        }

    def _discomfort(self, obs: Mapping) -> float:
        if "average_unmet_cooling_setpoint_difference" in obs:
            return max(float(obs["average_unmet_cooling_setpoint_difference"]), 0.0)
        if "indoor_dry_bulb_temperature_delta" in obs:
            return abs(float(obs["indoor_dry_bulb_temperature_delta"]))
        t = float(obs.get("indoor_dry_bulb_temperature", 0.0))
        s = float(obs.get("indoor_dry_bulb_temperature_set_point", t))
        return abs(t - s)

    def _wsum(self, c: Mapping) -> float:
        return float(sum(self.weights[k] * c[k] for k in self.weights))


class FlatSharedReward(ProjectReward):
    def __init__(self, env_metadata=None, **kw):
        super().__init__(env_metadata, weights=FLAT_WEIGHTS, shared=True, **kw)


class LocalIndividualReward(ProjectReward):
    def __init__(self, env_metadata=None, **kw):
        super().__init__(env_metadata, weights=FLAT_WEIGHTS, shared=False, **kw)


class UAEWeightedReward(ProjectReward):
    def __init__(self, env_metadata=None, **kw):
        super().__init__(env_metadata, weights=UAE_WEIGHTS, shared=True, **kw)


REWARD_FUNCTIONS = {
    "flat_shared":      FlatSharedReward,
    "local_individual": LocalIndividualReward,
    "uae_weighted":     UAEWeightedReward,
}


## 4 · MARL Environment Wrapper

Wraps `CityLearnEnv` into a parallel multi-agent interface.

Actions from the policy are in `[−1, 1]` and are mapped to each building's
native CityLearn action bounds. The `action_safety_factor` (default 0.75)
shrinks the effective range around the neutral point as required by the
CityLearn base paper, preventing unrealistic battery charge/discharge rates.


In [ ]:
def _to_box(space) -> gym.spaces.Box:
    return gym.spaces.Box(
        low=np.asarray(space.low, np.float32),
        high=np.asarray(space.high, np.float32),
        shape=space.shape,
        dtype=np.float32,
    )


def _neutral_action(space: gym.spaces.Box) -> np.ndarray:
    return np.clip(np.zeros(space.shape, np.float32), space.low, space.high)


def _parse_step(result):
    if len(result) == 5:
        return result
    obs, rews, done, info = result
    return obs, rews, bool(done), False, info


class CityLearnMARLEnv:
    def __init__(
        self,
        dataset_name: str = DATASET_NAME,
        reward_mode: str = "flat_shared",
        schema=None,
        seed: Optional[int] = None,
        action_safety_factor: float = 0.75,
    ):
        if reward_mode not in REWARD_FUNCTIONS:
            raise ValueError(f"Unknown reward_mode={reward_mode!r}")
        if not 0.0 < action_safety_factor <= 1.0:
            raise ValueError("action_safety_factor must be in (0, 1]")

        self.action_safety_factor = float(action_safety_factor)

        if schema is None:
            schema = DataSet().get_schema(dataset_name)

        self.base_env = CityLearnEnv(
            schema,
            reward_function=REWARD_FUNCTIONS[reward_mode],
            central_agent=False,
            random_seed=seed,
        )

        self.agents = [b.name for b in self.base_env.buildings]
        self._native_spaces = {
            a: _to_box(sp)
            for a, sp in zip(self.agents, self.base_env.action_space)
        }
        self.observation_spaces = {
            a: _to_box(sp)
            for a, sp in zip(self.agents, self.base_env.observation_space)
        }
        self.action_spaces = {
            a: gym.spaces.Box(-1.0, 1.0, shape=sp.shape, dtype=np.float32)
            for a, sp in self._native_spaces.items()
        }
        self._last_obs: Optional[Dict[str, np.ndarray]] = None

    def reset(self, seed: Optional[int] = None):
        if seed is not None:
            np.random.seed(seed)
        result = self.base_env.reset()
        obs, info = (result if isinstance(result, tuple) and len(result) == 2
                     else (result, {}))
        if hasattr(self.base_env.reward_function, "reset_memory"):
            self.base_env.reward_function.reset_memory()
        self._last_obs = self._fmt_obs(obs)
        return self._last_obs, {a: {} for a in self.agents}

    def step(self, actions: Dict[str, np.ndarray]):
        try:
            result = self.base_env.step(self._scale_actions(actions))
        except AssertionError:
            result = self.base_env.step(
                [_neutral_action(self._native_spaces[a]) for a in self.agents]
            )
        obs, rews, term, trunc, info = _parse_step(result)
        self._last_obs = self._fmt_obs(obs)
        rwd = {a: float(v) for a, v in zip(
            self.agents, rews if len(rews) > 1 else rews * len(self.agents)
        )}
        return (
            self._last_obs,
            rwd,
            {a: bool(term) for a in self.agents},
            {a: bool(trunc) for a in self.agents},
            {a: {} for a in self.agents},
        )

    def state(self) -> np.ndarray:
        return np.concatenate(
            [self._last_obs[a] for a in self.agents], dtype=np.float32
        )

    def kpis(self) -> pd.DataFrame:
        df = pd.DataFrame(self.base_env.evaluate())
        names = [
            "cost_total", "carbon_emissions_total",
            "daily_one_minus_load_factor_average",
            "discomfort_proportion", "ramping_average",
        ]
        if {"level", "cost_function", "value"}.issubset(df.columns):
            d = df[df["level"] == "district"]
            return d[d["cost_function"].isin(names)][
                ["cost_function", "value"]
            ].reset_index(drop=True)
        return df

    def close(self):
        fn = getattr(self.base_env, "close", None)
        if callable(fn):
            fn()

    def _fmt_obs(self, obs) -> Dict[str, np.ndarray]:
        return {
            a: np.asarray(
                list(o.values()) if isinstance(o, dict) else o, np.float32
            ).reshape(-1)
            for a, o in zip(self.agents, obs)
        }

    def _scale_actions(self, actions: Dict[str, np.ndarray]) -> List[np.ndarray]:
        out = []
        for a in self.agents:
            act = np.clip(np.asarray(actions[a], np.float32).reshape(-1), -1.0, 1.0)
            ns = self._native_spaces[a]
            neutral = _neutral_action(ns)
            pos_span = ns.high - neutral
            neg_span = neutral - ns.low
            scaled = np.where(act >= 0.0, act * pos_span, act * neg_span)
            native = neutral + self.action_safety_factor * scaled
            out.append(np.clip(native, ns.low, ns.high).astype(np.float32))
        return out


## 5 · PPO Actor/Critic, GAE, and Losses

In [ ]:
class RunningNormalize:
    def __init__(self, shape: int, eps: float = 1e-8, clip: float = 10.0):
        self.mean  = np.zeros(shape, np.float64)
        self.var   = np.ones(shape,  np.float64)
        self.count = 0
        self.eps   = eps
        self.clip  = clip

    def update(self, x: np.ndarray) -> None:
        x = np.asarray(x, np.float64)
        if x.ndim == 1:
            x = x[np.newaxis]
        n     = x.shape[0]
        total = self.count + n
        delta = x.mean(0) - self.mean
        self.mean += delta * n / total
        self.var  = (
            self.var * self.count
            + x.var(0) * n
            + delta ** 2 * self.count * n / total
        ) / total
        self.count = total

    def normalize(self, x: np.ndarray) -> np.ndarray:
        return np.clip(
            (x - self.mean) / np.sqrt(self.var + self.eps),
            -self.clip, self.clip,
        ).astype(np.float32)


def _layer_init(
    layer: nn.Linear,
    std: float = math.sqrt(2),
    bias: float = 0.0,
) -> nn.Linear:
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, bias)
    return layer


def _mlp(in_dim: int, out_dim: int, hidden: int, out_std: float = 1.0) -> nn.Sequential:
    return nn.Sequential(
        _layer_init(nn.Linear(in_dim, hidden)),
        nn.Tanh(),
        _layer_init(nn.Linear(hidden, hidden)),
        nn.Tanh(),
        _layer_init(nn.Linear(hidden, out_dim), std=out_std),
    )


def _atanh(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -0.999_999, 0.999_999)
    return 0.5 * (torch.log1p(x) - torch.log1p(-x))


class SquashedGaussianActor(nn.Module):
    def __init__(self, obs_dim: int, act_dim: int, hidden: int):
        super().__init__()
        self.mean_net = _mlp(obs_dim, act_dim, hidden, out_std=0.01)
        self.log_std  = nn.Parameter(torch.zeros(act_dim))

    def _dist(self, obs: torch.Tensor) -> Normal:
        return Normal(
            self.mean_net(obs),
            torch.exp(self.log_std).expand_as(self.mean_net(obs)),
        )

    def act(self, obs: torch.Tensor):
        d   = self._dist(obs)
        raw = d.rsample()
        a   = torch.tanh(raw)
        lp  = (d.log_prob(raw) - torch.log(1.0 - a.pow(2) + 1e-6)).sum(-1)
        return a, lp, d.entropy().sum(-1)

    def evaluate_actions(self, obs: torch.Tensor, a: torch.Tensor):
        raw = _atanh(a)
        d   = self._dist(obs)
        lp  = (d.log_prob(raw) - torch.log(1.0 - a.pow(2) + 1e-6)).sum(-1)
        return lp, d.entropy().sum(-1)

    def deterministic(self, obs: torch.Tensor) -> torch.Tensor:
        return torch.tanh(self.mean_net(obs))


class IPPOActorCritic(nn.Module):
    def __init__(self, obs_dim: int, act_dim: int, hidden: int):
        super().__init__()
        self.actor  = SquashedGaussianActor(obs_dim, act_dim, hidden)
        self.critic = _mlp(obs_dim, 1, hidden)

    def act_and_value(self, obs: torch.Tensor):
        a, lp, ent = self.actor.act(obs)
        v = self.critic(obs).squeeze(-1)
        return a, lp, ent, v

    def evaluate(self, obs: torch.Tensor, a: torch.Tensor):
        lp, ent = self.actor.evaluate_actions(obs, a)
        v = self.critic(obs).squeeze(-1)
        return lp, ent, v


class CentralCritic(nn.Module):
    def __init__(self, state_dim: int, hidden: int):
        super().__init__()
        self.net = _mlp(state_dim, 1, hidden)

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        return self.net(s).squeeze(-1)


def to_tensor(x) -> torch.Tensor:
    return torch.as_tensor(x, dtype=torch.float32, device=DEVICE)


def compute_gae(
    rewards: torch.Tensor,
    dones:   torch.Tensor,
    values:  torch.Tensor,
    last_val: torch.Tensor,
    gamma: float,
    lam:   float,
) -> Tuple[torch.Tensor, torch.Tensor]:
    T   = rewards.shape[0]
    adv = torch.zeros(T, device=DEVICE)
    g   = torch.zeros((), device=DEVICE)
    for t in reversed(range(T)):
        nv  = last_val if t == T - 1 else values[t + 1]
        nt  = 1.0 - dones[t]
        d   = rewards[t] + gamma * nv * nt - values[t]
        g   = d + gamma * lam * nt * g
        adv[t] = g
    return adv, adv + values


def ppo_losses(
    new_lp:  torch.Tensor,
    old_lp:  torch.Tensor,
    adv:     torch.Tensor,
    new_val: torch.Tensor,
    old_val: torch.Tensor,
    returns: torch.Tensor,
    clip:    float,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    ratio   = (new_lp - old_lp).exp()
    pg_loss = torch.max(
        -adv * ratio,
        -adv * torch.clamp(ratio, 1.0 - clip, 1.0 + clip),
    ).mean()
    val_clipped = old_val + torch.clamp(new_val - old_val, -clip, clip)
    vf_loss = 0.5 * torch.max(
        (new_val - returns).pow(2),
        (val_clipped - returns).pow(2),
    ).mean()
    approx_kl = ((ratio - 1.0) - (new_lp - old_lp)).mean().detach()
    return pg_loss, vf_loss, approx_kl


## 6 · Training Configuration

In [ ]:
@dataclass
class TrainConfig:
    dataset_name:         str   = DATASET_NAME
    seed:                 int   = SEED
    total_env_steps:      int   = 250_000
    rollout_steps:        int   = 1_024
    update_epochs:        int   = 10
    minibatch_size:       int   = 256
    gamma:                float = 0.99
    gae_lambda:           float = 0.95
    clip_coef:            float = 0.20
    ent_coef:             float = 0.01
    vf_coef:              float = 0.50
    max_grad_norm:        float = 0.50
    learning_rate:        float = 3e-4
    hidden_size:          int   = 256
    action_safety_factor: float = 0.75


cfg = TrainConfig()
set_seed(cfg.seed)

# Discover environment dimensions once
_env = CityLearnMARLEnv(seed=cfg.seed)
_obs, _ = _env.reset(seed=cfg.seed)
AGENTS    = _env.agents
OBS_DIM   = _obs[AGENTS[0]].shape[0]
ACT_DIM   = _env.action_spaces[AGENTS[0]].shape[0]
STATE_DIM = _env.state().shape[0]
_env.close()

print("Config:", cfg)
print(f"Agents: {AGENTS}")
print(f"obs_dim={OBS_DIM}  act_dim={ACT_DIM}  state_dim={STATE_DIM}")


## 7 · IPPO Training

Independent PPO: one actor-critic per building. Each policy receives only its
own local observation and is trained on its building's individual reward
(`local_individual`) or the shared district reward (`flat_shared`, `uae_weighted`).
Observation normalization (Welford running mean/variance) is applied before every
policy call. The value function uses a clipped loss to prevent large updates.


In [ ]:
def _collect_ippo_rollout(env, policies, obs, cfg, norms):
    bufs = {
        a: {k: [] for k in ("norm_obs", "actions", "logprobs",
                             "rewards",  "dones",   "values")}
        for a in env.agents
    }
    for _ in range(cfg.rollout_steps):
        actions = {}
        with torch.no_grad():
            for a in env.agents:
                norms[a].update(obs[a])
                no = norms[a].normalize(obs[a])
                ot = to_tensor(no).unsqueeze(0)
                act, lp, _, v = policies[a].act_and_value(ot)
                actions[a] = act.squeeze(0).cpu().numpy().astype(np.float32)
                bufs[a]["norm_obs"].append(no)
                bufs[a]["actions"].append(actions[a])
                bufs[a]["logprobs"].append(lp.item())
                bufs[a]["values"].append(v.item())

        next_obs, rews, terms, truncs, _ = env.step(actions)
        done = float(any(terms.values()) or any(truncs.values()))
        for a in env.agents:
            bufs[a]["rewards"].append(float(rews[a]))
            bufs[a]["dones"].append(done)
        obs = next_obs
        if done:
            obs, _ = env.reset()

    batches = {}
    with torch.no_grad():
        for a in env.agents:
            last_v = policies[a].critic(
                to_tensor(norms[a].normalize(obs[a])).unsqueeze(0)
            ).squeeze()
            rw = to_tensor(bufs[a]["rewards"])
            dn = to_tensor(bufs[a]["dones"])
            vs = to_tensor(bufs[a]["values"])
            adv, ret = compute_gae(rw, dn, vs, last_v, cfg.gamma, cfg.gae_lambda)
            batches[a] = {
                "norm_obs": to_tensor(np.asarray(bufs[a]["norm_obs"], np.float32)),
                "actions":  to_tensor(np.asarray(bufs[a]["actions"],  np.float32)),
                "logprobs": to_tensor(bufs[a]["logprobs"]),
                "values":   vs,
                "adv":      adv,
                "ret":      ret,
                "rewards":  rw,
            }
    return batches, obs


def _update_ippo(policies, optimizers, batches, cfg):
    rows = []
    for a, batch in batches.items():
        adv = (batch["adv"] - batch["adv"].mean()) / (batch["adv"].std() + 1e-8)
        bs  = batch["norm_obs"].shape[0]
        mb  = min(cfg.minibatch_size, bs)
        for _ in range(cfg.update_epochs):
            for start in range(0, bs, mb):
                idx = torch.randperm(bs, device=DEVICE)[start:start + mb]
                nlp, ent, nv = policies[a].evaluate(
                    batch["norm_obs"][idx], batch["actions"][idx]
                )
                pl, vl, kl = ppo_losses(
                    nlp, batch["logprobs"][idx], adv[idx],
                    nv, batch["values"][idx], batch["ret"][idx], cfg.clip_coef,
                )
                loss = pl + cfg.vf_coef * vl - cfg.ent_coef * ent.mean()
                optimizers[a].zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(policies[a].parameters(), cfg.max_grad_norm)
                optimizers[a].step()
        rows.append({
            "agent":          a,
            "policy_loss":    float(pl.detach()),
            "value_loss":     float(vl.detach()),
            "entropy":        float(ent.mean().detach()),
            "approx_kl":      float(kl),
            "rollout_reward": float(batch["rewards"].sum()),
        })
    return pd.DataFrame(rows)


def train_ippo(reward_mode: str, save_path: Path, out_dir: Path) -> Tuple:
    if save_path.exists():
        print(f"  [skip] {save_path.name} already exists.")
        ck    = torch.load(save_path, map_location=DEVICE, weights_only=False)
        pols  = nn.ModuleDict({
            a: IPPOActorCritic(OBS_DIM, ACT_DIM, cfg.hidden_size).to(DEVICE)
            for a in AGENTS
        })
        pols.load_state_dict(ck["state_dict"])
        norms = {a: RunningNormalize(OBS_DIM) for a in AGENTS}
        for a in AGENTS:
            norms[a].mean  = ck["norm_means"][a]
            norms[a].var   = ck["norm_vars"][a]
            norms[a].count = cfg.total_env_steps
        hist_f = out_dir / "training_history.csv"
        hist = pd.read_csv(hist_f) if hist_f.exists() else pd.DataFrame()
        return pols, norms, hist

    set_seed(cfg.seed)
    env = CityLearnMARLEnv(
        reward_mode=reward_mode, seed=cfg.seed,
        action_safety_factor=cfg.action_safety_factor,
    )
    obs, _ = env.reset(seed=cfg.seed)

    pols = nn.ModuleDict({
        a: IPPOActorCritic(OBS_DIM, ACT_DIM, cfg.hidden_size).to(DEVICE)
        for a in AGENTS
    })
    opts  = {a: optim.Adam(pols[a].parameters(), lr=cfg.learning_rate, eps=1e-5)
             for a in AGENTS}
    norms = {a: RunningNormalize(OBS_DIM) for a in AGENTS}

    n_updates = max(1, math.ceil(cfg.total_env_steps / cfg.rollout_steps))
    history   = []
    for upd in range(1, n_updates + 1):
        frac = 1.0 - (upd - 1) / n_updates
        for opt in opts.values():
            for pg in opt.param_groups:
                pg["lr"] = frac * cfg.learning_rate
        batches, obs = _collect_ippo_rollout(env, pols, obs, cfg, norms)
        df = _update_ippo(pols, opts, batches, cfg)
        history.append({
            "algorithm":          "ippo",
            "reward_mode":        reward_mode,
            "update":             upd,
            "env_steps":          upd * cfg.rollout_steps,
            "mean_rollout_reward":df["rollout_reward"].mean(),
            "mean_policy_loss":   df["policy_loss"].mean(),
            "mean_value_loss":    df["value_loss"].mean(),
            "mean_entropy":       df["entropy"].mean(),
        })
        if upd == 1 or upd % 20 == 0 or upd == n_updates:
            r = history[-1]
            print(f"  IPPO [{reward_mode}] {upd:03d}/{n_updates} "
                  f"| steps={r['env_steps']:>7} "
                  f"| reward={r['mean_rollout_reward']:>8.2f} "
                  f"| vf={r['mean_value_loss']:.3f} "
                  f"| lr={frac * cfg.learning_rate:.2e}")
    env.close()

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(out_dir / "training_history.csv", index=False)

    torch.save({
        "config":      vars(cfg),
        "agents":      AGENTS,
        "obs_dim":     OBS_DIM,
        "act_dim":     ACT_DIM,
        "reward_mode": reward_mode,
        "run_label":   RUN_LABEL,
        "seed":        SEED,
        "state_dict":  pols.state_dict(),
        "norm_means":  {a: norms[a].mean for a in AGENTS},
        "norm_vars":   {a: norms[a].var  for a in AGENTS},
    }, save_path)
    print(f"  Saved {save_path}")
    return pols, norms, hist_df


## 8 · MAPPO Training

Decentralised actors with a centralised critic (CTDE). The critic receives the
concatenated normalised observations from all buildings (180 dims) and is trained
on the mean team reward. Actors are trained independently on their local
observations using the same PPO objective as IPPO.


In [ ]:
def _norm_state(state, norms):
    parts = [norms[a].normalize(state[i * OBS_DIM:(i + 1) * OBS_DIM])
             for i, a in enumerate(AGENTS)]
    return np.concatenate(parts, dtype=np.float32)


def _collect_mappo_rollout(env, actors, critic, obs, cfg, norms):
    bufs = {
        "norm_states": [], "rewards": [], "dones": [], "values": [],
        "agent_nobs":  {a: [] for a in env.agents},
        "actions":     {a: [] for a in env.agents},
        "logprobs":    {a: [] for a in env.agents},
    }
    for _ in range(cfg.rollout_steps):
        raw_state = env.state()
        for a in env.agents:
            norms[a].update(obs[a])
        ns = _norm_state(raw_state, norms)

        with torch.no_grad():
            v = critic(to_tensor(ns).unsqueeze(0))
            bufs["norm_states"].append(ns)
            bufs["values"].append(v.item())
            actions = {}
            for a in env.agents:
                no = norms[a].normalize(obs[a])
                at, lp, _ = actors[a].act(to_tensor(no).unsqueeze(0))
                actions[a] = at.squeeze(0).cpu().numpy().astype(np.float32)
                bufs["agent_nobs"][a].append(no)
                bufs["actions"][a].append(actions[a])
                bufs["logprobs"][a].append(lp.item())

        next_obs, rews, terms, truncs, _ = env.step(actions)
        done = float(any(terms.values()) or any(truncs.values()))
        bufs["rewards"].append(float(np.mean(list(rews.values()))))
        bufs["dones"].append(done)
        obs = next_obs
        if done:
            obs, _ = env.reset()

    with torch.no_grad():
        last_v = critic(to_tensor(_norm_state(env.state(), norms)).unsqueeze(0)).squeeze()
        rw = to_tensor(bufs["rewards"])
        dn = to_tensor(bufs["dones"])
        vs = to_tensor(bufs["values"])
        adv, ret = compute_gae(rw, dn, vs, last_v, cfg.gamma, cfg.gae_lambda)

    return {
        "norm_states": to_tensor(np.asarray(bufs["norm_states"], np.float32)),
        "rewards": rw, "dones": dn, "values": vs, "adv": adv, "ret": ret,
        "agent_nobs": {a: to_tensor(np.asarray(bufs["agent_nobs"][a], np.float32))
                       for a in env.agents},
        "actions":    {a: to_tensor(np.asarray(bufs["actions"][a], np.float32))
                       for a in env.agents},
        "logprobs":   {a: to_tensor(bufs["logprobs"][a]) for a in env.agents},
    }, obs


def _update_mappo(actors, critic, optimizer, batch, cfg):
    adv = (batch["adv"] - batch["adv"].mean()) / (batch["adv"].std() + 1e-8)
    bs  = batch["norm_states"].shape[0]
    mb  = min(cfg.minibatch_size, bs)
    for _ in range(cfg.update_epochs):
        for start in range(0, bs, mb):
            idx = torch.randperm(bs, device=DEVICE)[start:start + mb]
            nv  = critic(batch["norm_states"][idx])
            _, vl, _ = ppo_losses(
                torch.zeros_like(nv), torch.zeros_like(nv), adv[idx],
                nv, batch["values"][idx], batch["ret"][idx], cfg.clip_coef,
            )
            pls, ents = [], []
            for a in actors:
                nlp, ent = actors[a].evaluate_actions(
                    batch["agent_nobs"][a][idx], batch["actions"][a][idx]
                )
                pl, _, _ = ppo_losses(
                    nlp, batch["logprobs"][a][idx], adv[idx],
                    nv.detach(), batch["values"][idx], batch["ret"][idx], cfg.clip_coef,
                )
                pls.append(pl); ents.append(ent.mean())
            pol_loss = torch.stack(pls).mean()
            ent_loss = torch.stack(ents).mean()
            loss = pol_loss + cfg.vf_coef * vl - cfg.ent_coef * ent_loss
            optimizer.zero_grad()
            loss.backward()
            params = list(critic.parameters()) + [
                p for ac in actors.values() for p in ac.parameters()
            ]
            nn.utils.clip_grad_norm_(params, cfg.max_grad_norm)
            optimizer.step()
    return {
        "policy_loss":    float(pol_loss.detach()),
        "value_loss":     float(vl.detach()),
        "entropy":        float(ent_loss.detach()),
        "rollout_reward": float(batch["rewards"].sum()),
    }


def train_mappo(reward_mode: str, save_path: Path, out_dir: Path) -> Tuple:
    if save_path.exists():
        print(f"  [skip] {save_path.name} already exists.")
        ck     = torch.load(save_path, map_location=DEVICE, weights_only=False)
        acts   = nn.ModuleDict({
            a: SquashedGaussianActor(OBS_DIM, ACT_DIM, cfg.hidden_size).to(DEVICE)
            for a in AGENTS
        })
        critic = CentralCritic(STATE_DIM, cfg.hidden_size).to(DEVICE)
        acts.load_state_dict(ck["actor_state_dict"])
        critic.load_state_dict(ck["critic_state_dict"])
        norms  = {a: RunningNormalize(OBS_DIM) for a in AGENTS}
        for a in AGENTS:
            norms[a].mean  = ck["norm_means"][a]
            norms[a].var   = ck["norm_vars"][a]
            norms[a].count = cfg.total_env_steps
        hist_f = out_dir / "training_history.csv"
        hist   = pd.read_csv(hist_f) if hist_f.exists() else pd.DataFrame()
        return acts, critic, norms, hist

    set_seed(cfg.seed + 1)
    env = CityLearnMARLEnv(
        reward_mode=reward_mode, seed=cfg.seed + 1,
        action_safety_factor=cfg.action_safety_factor,
    )
    obs, _ = env.reset(seed=cfg.seed + 1)

    actors = nn.ModuleDict({
        a: SquashedGaussianActor(OBS_DIM, ACT_DIM, cfg.hidden_size).to(DEVICE)
        for a in AGENTS
    })
    critic    = CentralCritic(STATE_DIM, cfg.hidden_size).to(DEVICE)
    optimizer = optim.Adam(
        list(critic.parameters()) +
        [p for ac in actors.values() for p in ac.parameters()],
        lr=cfg.learning_rate, eps=1e-5,
    )
    norms = {a: RunningNormalize(OBS_DIM) for a in AGENTS}

    n_updates = max(1, math.ceil(cfg.total_env_steps / cfg.rollout_steps))
    history   = []
    for upd in range(1, n_updates + 1):
        frac = 1.0 - (upd - 1) / n_updates
        for pg in optimizer.param_groups:
            pg["lr"] = frac * cfg.learning_rate
        batch, obs = _collect_mappo_rollout(env, actors, critic, obs, cfg, norms)
        m = _update_mappo(actors, critic, optimizer, batch, cfg)
        history.append({
            "algorithm":          "mappo",
            "reward_mode":        reward_mode,
            "update":             upd,
            "env_steps":          upd * cfg.rollout_steps,
            "mean_rollout_reward":m["rollout_reward"],
            "mean_policy_loss":   m["policy_loss"],
            "mean_value_loss":    m["value_loss"],
            "mean_entropy":       m["entropy"],
        })
        if upd == 1 or upd % 20 == 0 or upd == n_updates:
            r = history[-1]
            print(f"  MAPPO [{reward_mode}] {upd:03d}/{n_updates} "
                  f"| steps={r['env_steps']:>7} "
                  f"| reward={r['mean_rollout_reward']:>8.2f} "
                  f"| vf={r['mean_value_loss']:.3f} "
                  f"| lr={frac * cfg.learning_rate:.2e}")
    env.close()

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(out_dir / "training_history.csv", index=False)

    torch.save({
        "config":             vars(cfg),
        "agents":             AGENTS,
        "obs_dim":            OBS_DIM,
        "act_dim":            ACT_DIM,
        "state_dim":          STATE_DIM,
        "reward_mode":        reward_mode,
        "run_label":          RUN_LABEL,
        "seed":               SEED,
        "actor_state_dict":   actors.state_dict(),
        "critic_state_dict":  critic.state_dict(),
        "norm_means":  {a: norms[a].mean for a in AGENTS},
        "norm_vars":   {a: norms[a].var  for a in AGENTS},
    }, save_path)
    print(f"  Saved {save_path}")
    return actors, critic, norms, hist_df


## 9 · Train All 6 Conditions

Each condition is trained sequentially. If a checkpoint already exists for a
condition it is loaded and training is skipped, making the notebook safe to
resume after interruption.

**Expected runtime:** approximately 35 minutes per condition on CPU (6 × 35 ≈ 3.5 hours total).


In [ ]:
trained = {}

for algo, reward_mode in CONDITIONS:
    cp  = ckpt_path(algo, reward_mode)
    out = cond_dir(algo, reward_mode)
    print(f"\n{'='*60}")
    print(f"  {algo.upper()} | {reward_mode}")
    print(f"{'='*60}")

    if algo == "ippo":
        policies, norms, history = train_ippo(reward_mode, cp, out)
        trained[(algo, reward_mode)] = {
            "type": "ippo", "policies": policies, "norms": norms, "history": history
        }
    else:
        actors, critic, norms, history = train_mappo(reward_mode, cp, out)
        trained[(algo, reward_mode)] = {
            "type": "mappo", "actors": actors, "critic": critic,
            "norms": norms, "history": history
        }

print("\nAll 6 conditions complete.")
print("Checkpoints:")
for cp in sorted(MODELS_DIR.glob(f"*{CKPT_SUFFIX}.pt")):
    print(f"  {cp.name}  ({cp.stat().st_size // 1024} KB)")


## 10 · US-Climate Evaluation

All six trained policies are evaluated on a held-out rollout (`seed = SEED + 100`)
using the same US training climate. A neutral (zero-action) baseline is included
for comparison. All conditions are evaluated in the `local_individual` reward
environment so reward magnitudes are comparable across algorithms.


In [ ]:
EVAL_REWARD_MODE = "local_individual"
EVAL_SEED        = SEED + 100


@torch.no_grad()
def _get_actions(algo: str, reward_mode: str, obs: Dict) -> Dict:
    t = trained[(algo, reward_mode)]
    n = t["norms"]
    if t["type"] == "ippo":
        return {a: t["policies"][a].actor.deterministic(
                    to_tensor(n[a].normalize(obs[a])).unsqueeze(0)
                   ).squeeze(0).cpu().numpy().astype(np.float32)
                for a in AGENTS}
    return {a: t["actors"][a].deterministic(
                to_tensor(n[a].normalize(obs[a])).unsqueeze(0)
               ).squeeze(0).cpu().numpy().astype(np.float32)
            for a in AGENTS}


def evaluate_condition(
    algo: str,
    reward_mode: str,
    max_steps: int = 8760,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    env = CityLearnMARLEnv(
        reward_mode=EVAL_REWARD_MODE,
        seed=EVAL_SEED,
        action_safety_factor=cfg.action_safety_factor,
    )
    obs, _ = env.reset(seed=EVAL_SEED)
    cum = {a: 0.0 for a in AGENTS}
    for step in range(1, max_steps + 1):
        if algo == "neutral":
            acts = {a: np.zeros(env.action_spaces[a].shape, np.float32)
                    for a in AGENTS}
        else:
            acts = _get_actions(algo, reward_mode, obs)
        obs, rews, terms, truncs, _ = env.step(acts)
        for a, r in rews.items():
            cum[a] += float(r)
        if any(terms.values()) or any(truncs.values()):
            break
    total = float(sum(cum.values()))
    mean  = float(np.mean(list(cum.values())))
    label = "neutral" if algo == "neutral" else reward_mode
    try:
        kpis = env.kpis()
        kpis.insert(0, "reward_mode", label)
        kpis.insert(0, "algorithm",   algo)
    except Exception as exc:
        kpis = pd.DataFrame({"algorithm": [algo], "reward_mode": [label],
                             "error": [str(exc)]})
    env.close()
    return (
        pd.DataFrame([{"algorithm": algo, "reward_mode": label,
                       "steps": step, "total_reward": total,
                       "mean_agent_reward": mean, **cum}]),
        kpis,
    )


# Run evaluation
all_rewards, all_kpis = [], []
for label, algo, rm in [("neutral", "neutral", "neutral")] + [
    (f"{a}_{r}", a, r) for a, r in CONDITIONS
]:
    print(f"Evaluating {label} ...")
    rr, rk = evaluate_condition(algo, rm)
    all_rewards.append(rr)
    all_kpis.append(rk)
    (cond_dir(algo, rm) if algo != "neutral" else RESULTS_DIR / "conditions").mkdir(
        parents=True, exist_ok=True
    )
    if algo != "neutral":
        rr.to_csv(cond_dir(algo, rm) / "eval_reward.csv", index=False)
        rk.to_csv(cond_dir(algo, rm) / "eval_kpis.csv",   index=False)

rewards_df = pd.concat(all_rewards, ignore_index=True)
kpis_df    = pd.concat(all_kpis,    ignore_index=True)

summary_dir = RESULTS_DIR / "summary"
rewards_df.to_csv(summary_dir / "all_rewards.csv", index=False)
kpis_df.to_csv(summary_dir / "all_kpis.csv",       index=False)

print("\nEvaluation reward summary (higher = better):")
display(rewards_df[["algorithm", "reward_mode", "mean_agent_reward"]]
        .sort_values("mean_agent_reward", ascending=False))


## 11 · Plots and Summary CSVs

In [ ]:
KPI_NAMES = [
    "cost_total", "carbon_emissions_total",
    "discomfort_proportion", "ramping_average",
    "daily_one_minus_load_factor_average",
]

# Training history
all_hist = []
for algo, rm in CONDITIONS:
    h = trained[(algo, rm)]["history"]
    if not h.empty and "update" in h.columns:
        all_hist.append(h)
    else:
        p = cond_dir(algo, rm) / "training_history.csv"
        if p.exists():
            all_hist.append(pd.read_csv(p))

if all_hist:
    hist_all = pd.concat(all_hist, ignore_index=True)
    hist_all["condition"] = hist_all["algorithm"] + " | " + hist_all["reward_mode"]
    hist_all.to_csv(summary_dir / "all_training_history.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for cond, grp in hist_all.groupby("condition"):
        axes[0].plot(grp["env_steps"], grp["mean_rollout_reward"], lw=1.5, label=cond)
        axes[1].plot(grp["env_steps"], grp["mean_policy_loss"],    lw=1.5, label=cond)
        axes[2].plot(grp["env_steps"], grp["mean_value_loss"],     lw=1.5, label=cond)
    axes[0].set_title("Training reward");  axes[0].set_ylabel("Mean rollout reward")
    axes[1].set_title("Policy loss")
    axes[2].set_title("Value loss")
    for ax in axes:
        ax.set_xlabel("Environment steps"); ax.grid(alpha=0.3)
        ax.legend(fontsize=6)
    plt.suptitle(
        f"Training curves — {RUN_LABEL} (seed={SEED})", fontsize=12, y=1.01
    )
    plt.tight_layout()
    plt.savefig(summary_dir / "training_curves_all.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    display(Image(str(summary_dir / "training_curves_all.png")))

# KPI comparison
if {"cost_function", "value", "algorithm"}.issubset(kpis_df.columns):
    kpi_pivot = kpis_df.pivot_table(
        index="cost_function",
        columns=["algorithm", "reward_mode"],
        values="value",
    )
    kpi_pivot.to_csv(summary_dir / "kpi_pivot.csv")

    # % change vs neutral
    neutral_kpis = kpis_df[kpis_df["algorithm"] == "neutral"].set_index(
        "cost_function"
    )["value"]
    rows = []
    for algo, rm in CONDITIONS:
        sub = kpis_df[(kpis_df["algorithm"] == algo) & (kpis_df["reward_mode"] == rm)]
        row = {"algorithm": algo, "reward_mode": rm}
        for kpi in KPI_NAMES:
            v = sub[sub["cost_function"] == kpi]["value"]
            b = neutral_kpis.get(kpi, np.nan)
            if len(v) and not np.isnan(b):
                val = float(v.iloc[0])
                row[kpi] = round(val, 4)
                row[f"{kpi}_pct_vs_neutral"] = round((val - b) / abs(b) * 100, 2)
        rows.append(row)
    pct_df = pd.DataFrame(rows)
    pct_df.to_csv(summary_dir / "pct_change_vs_neutral.csv", index=False)

    # Evaluation bar chart
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    plot_df = rewards_df.sort_values("mean_agent_reward", ascending=False).copy()
    plot_df["label"] = plot_df["algorithm"] + "|" + plot_df["reward_mode"]
    colors = ["#cccccc" if "neutral" in r else
              "#4C78A8" if r == "ippo"  else "#F58518"
              for r in plot_df["algorithm"]]
    ax2.bar(plot_df["label"], plot_df["mean_agent_reward"], color=colors, alpha=0.9)
    baseline = float(
        rewards_df[rewards_df["algorithm"] == "neutral"]["mean_agent_reward"].iloc[0]
    )
    ax2.axhline(baseline, color="red", lw=1.5, ls="--", label="neutral baseline")
    ax2.set_title(f"Evaluation reward — {RUN_LABEL} (seed={SEED})")
    ax2.set_ylabel("Mean agent reward")
    ax2.tick_params(axis="x", rotation=30, labelsize=8)
    ax2.legend(); ax2.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(summary_dir / "reward_comparison_all.png", dpi=150, bbox_inches="tight")
    plt.close(fig2)
    display(Image(str(summary_dir / "reward_comparison_all.png")))

    # KPI grouped bar chart
    plot_kpis = kpis_df[kpis_df["cost_function"].isin(KPI_NAMES)].copy()
    plot_kpis["condition"] = plot_kpis["algorithm"] + "|" + plot_kpis["reward_mode"]
    fig3, axes3 = plt.subplots(1, len(KPI_NAMES), figsize=(22, 5))
    for ax, kpi in zip(axes3, KPI_NAMES):
        sub = plot_kpis[plot_kpis["cost_function"] == kpi].sort_values("value")
        ax.barh(sub["condition"], sub["value"], alpha=0.85)
        ax.axvline(
            float(neutral_kpis.get(kpi, np.nan)),
            color="red", lw=1, ls="--", alpha=0.7,
        )
        ax.set_title(kpi.replace("_", " "), fontsize=8)
        ax.grid(axis="x", alpha=0.3)
        ax.tick_params(axis="y", labelsize=7)
    plt.suptitle(f"KPI comparison — {RUN_LABEL} (lower = better)", fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig(summary_dir / "kpi_comparison_all.png", dpi=150, bbox_inches="tight")
    plt.close(fig3)
    display(Image(str(summary_dir / "kpi_comparison_all.png")))

    print("Summary tables saved.")
    display(pct_df[["algorithm", "reward_mode"] +
                   [f"{k}_pct_vs_neutral" for k in KPI_NAMES
                    if f"{k}_pct_vs_neutral" in pct_df.columns]].round(2))


## 12 · Output Manifest

`manifest.json` records all experiment metadata needed for reproducibility.


In [ ]:
import datetime
import importlib

def _pkg_version(name: str) -> str:
    try:
        return importlib.import_module(name).__version__
    except Exception:
        return "?"


manifest = {
    "run_label":        RUN_LABEL,
    "run_index":        RUN_INDEX,
    "seed":             SEED,
    "total_env_steps":  cfg.total_env_steps,
    "dataset":          DATASET_NAME,
    "conditions":       [f"{a}_{r}" for a, r in CONDITIONS],
    "eval_reward_mode": EVAL_REWARD_MODE,
    "device":           str(DEVICE),
    "python_version":   sys.version,
    "timestamp":        datetime.datetime.utcnow().isoformat() + "Z",
    "packages": {
        "citylearn":  _pkg_version("citylearn"),
        "torch":      _pkg_version("torch"),
        "numpy":      _pkg_version("numpy"),
        "pandas":     _pkg_version("pandas"),
        "gymnasium":  _pkg_version("gymnasium"),
        "matplotlib": _pkg_version("matplotlib"),
    },
    "hyperparameters": vars(cfg),
}

import json
manifest_path = RESULTS_DIR / "manifest.json"
with open(manifest_path, "w") as fp:
    json.dump(manifest, fp, indent=2, default=str)

print(f"Manifest written to {manifest_path}")
print(json.dumps(manifest, indent=2, default=str))


## 13 · Bundle Artifacts

Packages all results and model checkpoints for this run into a single zip file
for archival and sharing. The `.pt` checkpoint files are large (2–4 MB each)
and are excluded from the GitHub repository by `.gitignore`; they are included
in the zip for offline storage only.


In [ ]:
import zipfile

zip_path = ARTIFACTS_DIR / f"citylearn_marl_{RUN_LABEL}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    # results directory
    for f in sorted(RESULTS_DIR.rglob("*")):
        if f.is_file():
            zf.write(f, f.relative_to(RESULTS_DIR.parent))
    # model checkpoints for this run
    for pt in sorted(MODELS_DIR.glob(f"*{CKPT_SUFFIX}.pt")):
        zf.write(pt, pt)

size_mb = zip_path.stat().st_size / 1_048_576
print(f"Artifact bundle: {zip_path}  ({size_mb:.1f} MB)")
print("Contents:")
with zipfile.ZipFile(zip_path) as zf:
    for name in sorted(zf.namelist()):
        info = zf.getinfo(name)
        print(f"  {name}  ({info.file_size // 1024} KB)")
